In [1]:
import pandas as pd
import numpy as np 
import cv2
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.utils import class_weight
import os

In [2]:
# meta/images paths
def map_imgs(meta_file_path, image_file_path):
    IMAGE_PATH = image_file_path
#     meta_file= '/kaggle/input/siim-isic-melanoma-classification/train.csv'
#     IMAGE_PATH = '/kaggle/input/siim-isic-melanoma-classification/jpeg/train'
    
    df = pd.read_csv(meta_file_path)

    # adding new column = filepath (complete image path)
    df['image_path'] = df['image_name'].map(lambda x:  os.path.join(IMAGE_PATH,x+'.jpg'))

    # mapping dictionary
    labelmap = {}
    for l in df.target.unique().tolist():
        if l == 1:
            labelmap[l] = 'melanoma'
        else:
            labelmap[l] = 'benign'

    # seperate list of image that are labelled = melanoma
    df_melanoma = df[df['target'] == 1]
    df_melanoma.reset_index(drop=True, inplace=True)

    # view 
    print(f"Found {df.groupby('target').count()['image_name'][1]} images that are labelled = melanoma and {df.groupby('target').count()['image_name'][0]} that are labelled = benign")
    return df

In [3]:
df_train = map_imgs("/kaggle/input/siim-isic-melanoma-classification/train.csv","/kaggle/input/siim-isic-melanoma-classification/jpeg/train")
df_train= df_train[["target","image_path"]]
df_train["target"] = df_train["target"].astype(str)
df_train

Found 584 images that are labelled = melanoma and 32542 that are labelled = benign


,target,image_path
0,0,/kaggle/input/siim-isic-melanoma-classificatio...
1,0,/kaggle/input/siim-isic-melanoma-classificatio...
2,0,/kaggle/input/siim-isic-melanoma-classificatio...
3,0,/kaggle/input/siim-isic-melanoma-classificatio...
4,0,/kaggle/input/siim-isic-melanoma-classificatio...
...,...,...
33121,0,/kaggle/input/siim-isic-melanoma-classificatio...
33122,0,/kaggle/input/siim-isic-melanoma-classificatio...
33123,0,/kaggle/input/siim-isic-melanoma-classificatio...
33124,0,/kaggle/input/siim-isic-melanoma-classificatio...


In [4]:
data_generator = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)

In [5]:
train_generator = data_generator.flow_from_dataframe(df_train,
                                              target_size= (224, 224),
                                                     x_col='image_path',
                                                     y_col='target',
                                              batch_size = 128,
                                              subset = 'training',
                                              shuffle = True,
                                              class_mode ='binary')
val_generator = data_generator.flow_from_dataframe(df_train,
                                              target_size= (224, 224),x_col='image_path',
                                                     y_col='target',
                                              batch_size = 128,
                                              subset = 'validation',
                                              shuffle = False,
                                              class_mode ='binary')

Found 26501 validated image filenames belonging to 2 classes.
Found 6625 validated image filenames belonging to 2 classes.


In [6]:
class_weights = class_weight.compute_class_weight('balanced',
                                                 classes= np.unique(df_train["target"]),
                                                 y =df_train["target"])

In [7]:
# convert numpy array to dictionary
class_weight = dict(enumerate(class_weights.flatten(), 0))
class_weight

{0: 0.5089730194825149, 1: 28.361301369863014}

In [8]:
base_model = ResNet50(include_top=False, pooling='avg', weights='imagenet', input_shape=(224,224,3))
for layer in base_model.layers[:-4]:
    layer.trainable = False
# base_model.summary() 


94765736/94765736 [==============================] - 1s 0us/step


In [9]:
STEPS_PER_EPOCH = 26501 // 128
VALID_STEPS = 6625 // 128

In [10]:
my_model = Sequential([base_model])
# my_model.add(GlobalAveragePooling2D())
my_model.add(Dense(512, activation='relu'))
my_model.add(Dropout(0.5))
my_model.add(Dense(1, activation='sigmoid'))

my_model.compile(optimizer=Adam(learning_rate=0.0001),
             loss='binary_crossentropy',
             metrics=['accuracy',tf.keras.metrics.Recall()])

model_name = "melanoma_model.h5"
checkpoint = ModelCheckpoint(model_name,
                            monitor="val_loss",
                            mode="min",
                            save_best_only = True,
                            verbose=1)

earlystopping = EarlyStopping(monitor='val_loss',min_delta = 0, patience = 5, verbose = 1, restore_best_weights=True)

try:
    history = my_model.fit(train_generator,
                           epochs=5,
                           steps_per_epoch=STEPS_PER_EPOCH,
                           validation_data=val_generator,
                           validation_steps=VALID_STEPS,
                           callbacks=[checkpoint,earlystopping],
                           class_weight=class_weight)
except KeyboardInterrupt:
    print("\nTraining Stopped")

Epoch 1/5
207/207 [==============================] - ETA: 0s - loss: 0.6126 - accuracy: 0.7208 - recall: 0.7284 
Epoch 1: val_loss improved from inf to 0.56520, saving model to melanoma_model.h5
207/207 [==============================] - 5978s 29s/step - loss: 0.6126 - accuracy: 0.7208 - recall: 0.7284 - val_loss: 0.5652 - val_accuracy: 0.7319 - val_recall: 0.7788
Epoch 2/5
207/207 [==============================] - ETA: 0s - loss: 0.4900 - accuracy: 0.7640 - recall: 0.7824 
Epoch 2: val_loss improved from 0.56520 to 0.53150, saving model to melanoma_model.h5
207/207 [==============================] - 5818s 28s/step - loss: 0.4900 - accuracy: 0.7640 - recall: 0.7824 - val_loss: 0.5315 - val_accuracy: 0.7466 - val_recall: 0.8173
Epoch 3/5
207/207 [==============================] - ETA: 0s - loss: 0.4268 - accuracy: 0.7896 - recall: 0.8214 
Epoch 3: val_loss improved from 0.53150 to 0.44891, saving model to melanoma_model.h5
207/207 [==============================] - 5997s 29s/step - los